# Actividad final — Mini proyecto de análisis de datos

**Entrega:** viernes 13 de noviembre de 2026  
**Título:** Exploración didáctica de temperatura superficial del mar y clorofila

Este mini proyecto utiliza datos **simulados de ejemplo** mientras no se incorpore el conjunto oficial de la organización. No constituyen evidencia oceanográfica real.

**Pregunta:** ¿Cómo varían descriptivamente la temperatura y la clorofila en seis meses de ejemplo?

## Preparación del entorno
Esta celda comprueba si faltan las bibliotecas necesarias y, solo en ese caso, instala el archivo `requirements.txt` de la raíz del repositorio en el kernel activo. Requiere conexión a Internet durante la primera instalación. Si instala paquetes, reinicia el kernel antes de continuar.


In [ ]:
from pathlib import Path
import importlib.util

# Buscar requirements.txt desde el directorio de trabajo actual
# (VS Code puede ejecutar desde la raíz o desde la carpeta de la actividad).
requirements = next(
    (carpeta / "requirements.txt"
     for carpeta in [Path.cwd(), *Path.cwd().parents]
     if (carpeta / "requirements.txt").is_file()),
    None,
)
if requirements is None:
    raise FileNotFoundError("No se encontró requirements.txt. Abre el notebook dentro del repositorio preOHW2026.")

faltantes = [modulo for modulo in ("pandas", "matplotlib")
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    print(f"Faltan {', '.join(faltantes)}. Instalando dependencias de {requirements}...")
    # %pip instala paquetes en el Python que corresponde al kernel del notebook.
    get_ipython().run_line_magic("pip", f'install -r "{requirements}"')
    importlib.invalidate_caches()
else:
    print("Dependencias pandas y matplotlib disponibles.")
print("Entorno preparado. Si acabas de instalar paquetes, reinicia el kernel y vuelve a ejecutar todas las celdas.")

## 1. Carga de datos y validación

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ruta = Path("datos/datos_ejemplo.csv")
if not ruta.exists():
    ruta = Path("03_proyecto_final/datos/datos_ejemplo.csv")
if not ruta.exists():
    raise FileNotFoundError("No se encuentra el CSV del mini proyecto")
df = pd.read_csv(ruta)
requeridas = {"mes", "temperatura_c", "clorofila_mg_m3"}
assert requeridas.issubset(df.columns)
print(f"Registros: {len(df)}; columnas: {df.shape[1]}")
display(df.head())

## 2. Exploración básica

In [ ]:
print("Valores ausentes por variable:")
print(df.isna().sum())
display(df[["temperatura_c", "clorofila_mg_m3"]].describe())

## 3. Función propia documentada

In [ ]:
def amplitud_observada(valores):
    """Calcula la diferencia entre el máximo y mínimo de valores válidos.

    Parámetros
    ----------
    valores : secuencia numérica
        Serie cuyos valores pueden incluir datos ausentes.

    Retorna
    -------
    float
        Amplitud (máximo - mínimo) o NaN sin valores válidos.
    """
    serie = pd.to_numeric(pd.Series(valores), errors="coerce").dropna()
    return float(serie.max() - serie.min()) if not serie.empty else float("nan")

print("Amplitud TSM (°C):", amplitud_observada(df["temperatura_c"]))
print("Amplitud clorofila (mg/m³):", amplitud_observada(df["clorofila_mg_m3"]))
assert amplitud_observada([2, 5, None, 9]) == 7.0

## 4. Visualización y exportación

Una figura con dos paneles permite comparar los cambios mensuales sin confundir las unidades. La figura se guarda en `figuras/`.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axs[0].plot(df["mes"], df["temperatura_c"], marker="o")
axs[0].set(ylabel="TSM (°C)", title="Variación mensual (datos simulados)")
axs[1].plot(df["mes"], df["clorofila_mg_m3"], marker="s")
axs[1].set(xlabel="Mes", ylabel="Clorofila (mg/m³)")
for ax in axs:
    ax.grid(alpha=0.3)
fig.tight_layout()
carpeta_fig = ruta.parent.parent / "figuras"
carpeta_fig.mkdir(exist_ok=True)
salida = carpeta_fig / "variacion_mensual.png"
fig.savefig(salida, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura exportada: {salida}")

## 5. Conclusiones breves

1. El ejemplo permite observar variaciones mensuales de ambas variables dentro del período representado.
2. La función `amplitud_observada()` resume el rango de variabilidad de cada serie.
3. **Limitación:** los datos son simulados y corresponden a solo seis meses; no pueden sustentar conclusiones ecológicas o estadísticas sobre el océano real.

## 6. Siguiente paso antes de la evaluación

Sustituir el CSV por el conjunto oficial entregado por la organización, adaptar el análisis y actualizar las conclusiones, si ese archivo difiere del ejemplo. Ejecutar todas las celdas, guardar el notebook y registrar el commit final.